In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import os

from marmopose.config import Config
from marmopose.utils.data_io import load_points_3d_h5


In [ ]:
config_path = '../configs/default.yaml'

config = Config(
    config_path=config_path,
    
    n_tracks=1,
    project='../demos/single',
)
idx_spinemid = config.animal['bodyparts'].index('spinemid')
idx_tailbase = config.animal['bodyparts'].index('tailbase')
idx_neck = config.animal['bodyparts'].index('neck')


In [ ]:
video = 'TestHome6.1'
video_path = f"/srv/MarmOT/VideoTracking/Videos/{video}/Input"
points_3d = load_points_3d_h5(f"/scratch/VideoTracking/Videos/{video}/Output_basemodel/points_3d/optimized.h5")
f = 1500

In [ ]:
%matplotlib inline
plt.clf()
fig, ax = plt.subplots(figsize=(16, 4))
ax.axis('off')
norm = mcolors.Normalize(vmin=-20, vmax=20)
data = points_3d[0,f + 1,:,:] - points_3d[0,f,:,:]
data = np.concatenate((data,np.mean(data,axis=0,keepdims=True)),axis=0)
data = np.concatenate((data,np.sqrt(np.einsum("ij,ij->i", data, data))[:,np.newaxis]),axis=1).transpose((1,0))
print(data.shape)
cmap = plt.cm.RdYlBu
table = ax.table(
    cellText=np.round(data,decimals=2),
    cellColours= cmap(norm(data)),
    rowLabels=['Velocity X','Velocity Y','Velocity Z','Velocity magnitude'],
    colLabels=config.animal['bodyparts'] + ['Mean'],
    loc='center',
    cellLoc='center',
    fontsize = 20
)
plt.show()

fig, axs = plt.subplots(2,2,figsize=(15,10))
axs = axs.flatten()
for i in range(1,5):
    vidcap = cv2.VideoCapture(os.path.join(video_path,f'output{i}.mp4'))
    vidcap.set(cv2.CAP_PROP_POS_FRAMES, f)
    success, frame = vidcap.read()
    if not success:
        print(f"Couldn't read frame {f} in video {os.path.join(video_path,f'output{i}.mp4')}")
    axs[i-1].imshow(frame[..., ::-1])
    axs[i-1].axis('off')
    axs[i-1].set_title(f'Frame {f}')
fig.show()

fig, axs = plt.subplots(2,2,figsize=(15,10))
axs = axs.flatten()
for i in range(1,5):
    vidcap = cv2.VideoCapture(os.path.join(video_path,f'output{i}.mp4'))
    vidcap.set(cv2.CAP_PROP_POS_FRAMES, f + 1)
    success, frame = vidcap.read()
    if not success:
        print(f"Couldn't read frame {f + 1} in video {os.path.join(video_path,f'output{i}.mp4')}")
    axs[i-1].imshow(frame[..., ::-1])
    axs[i-1].axis('off')
    axs[i-1].set_title(f'Frame {f + 1}')
fig.show()



In [ ]:

%matplotlib widget
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
for bodyparts in config.visualization['skeleton'][::-1]:
    idx_bodyparts = []
    for bodypart in bodyparts:
        idx_bodyparts.append(config.animal['bodyparts'].index(bodypart))
    ax.plot(points_3d[0,f,idx_bodyparts,0],points_3d[0,f,idx_bodyparts,1],points_3d[0,f,idx_bodyparts,2], marker = 'o', ms=3,c='b')
    ax.plot(points_3d[0,f+1,idx_bodyparts,0],points_3d[0,f+1,idx_bodyparts,1],points_3d[0,f+1,idx_bodyparts,2], marker = 'o', ms=3,c='r')
fig.show()
